In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import numpy as np
import albumentations as A
from albumentations.pytorch import ToTensorV2
import time # To time epochs
from tqdm import tqdm # For progress bars

In [2]:
OUTPUT_DIR = 'data_processed'

In [3]:
import torch
import gc

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()
    print(f"GPU memory cleared. Available: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

GPU memory cleared. Available: 8.00 GB


In [4]:
import os
import numpy as np
import torch
from torch.utils.data import Dataset

class LCTSC_3D_Dataset(Dataset):
    def __init__(self, data_dir, target_depth=128, image_transform=None, mask_transform=None, validation=False):
        self.data_dir = data_dir
        self.image_transform = image_transform
        self.mask_transform = mask_transform
        self.target_depth = target_depth
        self.is_validation = validation

        self.data_cache = {}  # Load volumes in memory
        self.files = []

        print("Loading dataset into memory...")
        for file_name in sorted(os.listdir(data_dir)):
            if file_name.endswith('.npz'):
                file_path = os.path.join(data_dir, file_name)
                try:
                    npz_file = np.load(file_path)
                    self.data_cache[file_path] = {
                        'image': npz_file['image'].astype(np.float32),
                        'Lung_R': npz_file['Lung_R'].astype(np.float32),
                        'Lung_L': npz_file['Lung_L'].astype(np.float32)
                    }
                    npz_file.close()
                    self.files.append(file_path)
                    print(f"  ✓ Loaded {file_name}: volume shape = {self.data_cache[file_path]['image'].shape}")
                except Exception as e:
                    print(f"  ✗ Error loading {file_name}: {e}")

        print(f"Total volumes loaded: {len(self.files)}\n")

    def __len__(self):
        return len(self.files)

    def pad_or_crop_depth(self, volume):
        """Pad or crop volume to target_depth along depth axis."""
        H, W, D = volume.shape
        if D == self.target_depth:
            return volume
        elif D > self.target_depth:
            start = (D - self.target_depth) // 2
            return volume[:, :, start:start+self.target_depth]
        else:
            pad_before = (self.target_depth - D) // 2
            pad_after = self.target_depth - D - pad_before
            return np.pad(volume, ((0,0), (0,0), (pad_before, pad_after)), mode='constant', constant_values=0)

    def __getitem__(self, idx):
        file_path = self.files[idx]
        data = self.data_cache[file_path]

        # Combine masks
        mask = np.maximum(data['Lung_R'], data['Lung_L'])

        # Pad/crop depth
        image = self.pad_or_crop_depth(data['image'])
        mask  = self.pad_or_crop_depth(mask)

        # Optional transforms
        if self.image_transform:
            image = self.image_transform(image=image)['image']
        else:
            image = torch.from_numpy(image).float()

        if self.mask_transform:
            mask = self.mask_transform(image=mask)['image']
        else:
            mask = torch.from_numpy(mask).float()

        image = image.permute(2, 0, 1)  # (H, W, D) -> (D, H, W)
        mask  = mask.permute(2, 0, 1)
        # Reorder to (C, D, H, W)
        image = image.unsqueeze(0) if image.dim() == 3 else image  # (1, D, H, W) -> (1, D, H, W)
        mask  = mask.unsqueeze(0)  if mask.dim() == 3 else mask


        return image, mask

In [5]:
TRAIN_IMAGE_DIR = os.path.join(OUTPUT_DIR, 'Train')
VAL_IMAGE_DIR = os.path.join(OUTPUT_DIR, 'Test')

batch_size = 4
train_dataset = LCTSC_3D_Dataset(data_dir=TRAIN_IMAGE_DIR, validation=False)
val_dataset = LCTSC_3D_Dataset(data_dir=VAL_IMAGE_DIR, validation=True)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, pin_memory=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, pin_memory=True, num_workers=4)

print(f"Train dataset: {len(train_dataset)} volumes")
print(f"Val dataset: {len(val_dataset)} volumes")

if train_loader:
    print("\nChecking one batch from train_loader...")
    try:
        images, masks = next(iter(train_loader))
        print(f"Image batch shape: {images.shape}, range: [{images.min():.3f}, {images.max():.3f}]")
        print(f"Mask batch shape: {masks.shape}, range: [{masks.min():.3f}, {masks.max():.3f}]")
        print("Successfully loaded one batch from train_loader.")
    except Exception as e:
        print(f"Error loading batch from train_loader: {e}")


if val_loader:
    print("\nChecking one batch from val_loader...")
    try:
        images, masks = next(iter(val_loader))
        print(f"Image batch shape: {images.shape}, range: [{images.min():.3f}, {images.max():.3f}]")
        print(f"Mask batch shape: {masks.shape}, range: [{masks.min():.3f}, {masks.max():.3f}]")
        print("Successfully loaded one batch from val_loader.")
    except Exception as e:
        print(f"Error loading batch from val_loader: {e}")

Loading dataset into memory...
  ✓ Loaded LCTSC-Train-S1-001.npz: volume shape = (128, 128, 109)
  ✓ Loaded LCTSC-Train-S1-002.npz: volume shape = (128, 128, 118)
  ✓ Loaded LCTSC-Train-S1-003.npz: volume shape = (128, 128, 124)
  ✓ Loaded LCTSC-Train-S1-004.npz: volume shape = (128, 128, 115)
  ✓ Loaded LCTSC-Train-S1-005.npz: volume shape = (128, 128, 100)
  ✓ Loaded LCTSC-Train-S1-006.npz: volume shape = (128, 128, 99)
  ✓ Loaded LCTSC-Train-S1-007.npz: volume shape = (128, 128, 102)
  ✓ Loaded LCTSC-Train-S1-008.npz: volume shape = (128, 128, 98)
  ✓ Loaded LCTSC-Train-S1-009.npz: volume shape = (128, 128, 122)
  ✓ Loaded LCTSC-Train-S1-010.npz: volume shape = (128, 128, 87)
  ✓ Loaded LCTSC-Train-S1-011.npz: volume shape = (128, 128, 112)
  ✓ Loaded LCTSC-Train-S1-012.npz: volume shape = (128, 128, 102)
  ✓ Loaded LCTSC-Train-S2-001.npz: volume shape = (128, 128, 132)
  ✓ Loaded LCTSC-Train-S2-002.npz: volume shape = (128, 128, 124)
  ✓ Loaded LCTSC-Train-S2-003.npz: volume shape 

In [6]:
import segmentation_models_pytorch_3d as smp3d
model = smp3d.Unet(
    encoder_name="resnet18",        # choose encoder, e.g. mobilenet_v2 or efficientnet-b7
    encoder_weights=None,
    in_channels=1,                  # model input channels (1 for grayscale)
    classes=1,                  # model output channels (number of classes in your dataset)
)

In [7]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [8]:
# Training loop with pytorch

loss_fn = smp3d.losses.DiceLoss(mode='binary')
optimizer = optim.Adam(model.parameters(), lr=1e-4)
num_epochs = 10
#lr_scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs * len(train_loader))

In [12]:
from tqdm.notebook import tqdm

# Create GradScaler for mixed precision training
scaler = torch.amp.GradScaler("cuda")

def train_one_epoch(model, dataloader, loss_fn, optimizer, device, scaler):
    model.train()
    running_loss = 0.0
    total_samples = 0

    pbar = tqdm(dataloader, desc="Training", leave=True)
    for images, masks in pbar:
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type="cuda"):
            outputs = model(images)
            loss = loss_fn(outputs, masks)

        # Use scaler for mixed precision
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        batch_size = images.size(0)
        running_loss += loss.item() * batch_size
        total_samples += batch_size
        avg_loss = running_loss / total_samples  # Fixed: divide by samples, not batches
        
        pbar.set_postfix({
            'avg_loss': f'{avg_loss:.6f}',
            'grad_norm': f'{grad_norm:.4f}',
            'out_range': f'[{outputs.min():.2f}, {outputs.max():.2f}]'
        }, refresh=True)

    epoch_loss = running_loss / len(dataloader.dataset)  # Fixed: use dataset length
    return epoch_loss

In [13]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [14]:
print(f"Using device: {device}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"Total GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB" if torch.cuda.is_available() else "CPU device")

if torch.cuda.is_available():
    torch.cuda.empty_cache()

model.to(device)

# Count model parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nModel: UNet with ResNet18 encoder")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Loss function: Dice Loss (Binary mode)")
print(f"Optimizer: Adam (lr=1e-4)")
print(f"Batch size: {batch_size} volumes")
print(f"Training samples: {len(train_dataset)} volumes")
print(f"Validation samples: {len(val_dataset)} volumes")
print()

# Test first batch to debug
print("="*80)
print("Testing first batch...")
print("="*80)
test_images, test_masks = next(iter(train_loader))
test_images = test_images.to(device).float()
test_masks = test_masks.to(device).float()

with torch.autocast(device_type="cuda"):
    test_outputs = model(test_images)
    test_loss = loss_fn(test_outputs, test_masks)

print(f"Input shape: {test_images.shape}")
print(f"Input range: [{test_images.min():.3f}, {test_images.max():.3f}]")
print(f"\nMask shape: {test_masks.shape}")
print(f"Mask range: [{test_masks.min():.3f}, {test_masks.max():.3f}]")
print(f"Mask positive pixels: {(test_masks > 0.5).sum().item()} / {test_masks.numel()} ({100 * (test_masks > 0.5).sum().item() / test_masks.numel():.2f}%)")
print(f"\nOutput shape: {test_outputs.shape}")
print(f"Output range: [{test_outputs.min():.3f}, {test_outputs.max():.3f}]")
print(f"Output mean: {test_outputs.mean():.3f}")
print(f"\nTest loss: {test_loss.item():.6f}")
print("="*80)
print()

# Training loop
best_val_loss = float('inf')
train_losses = []
val_losses = []

for epoch in range(num_epochs):
    print(f"\n{'='*80}")
    print(f"Epoch {epoch + 1}/{num_epochs}")
    print(f"{'='*80}")

    start_time = time.time()

    # Training
    print("\n[TRAINING]")
    train_loss = train_one_epoch(model, train_loader, loss_fn, optimizer, device, scaler)
    train_losses.append(train_loss)

    print(f"Epoch train loss: {train_loss:.6f}")
    print(f"Learning rate: {optimizer.param_groups[0]['lr']:.2e}")

    # Validation
    print("\n[VALIDATION]")
    model.eval()
    running_vloss = 0.0
    val_batch_losses = []

    with torch.no_grad():
        for vimages, vmasks in tqdm(val_loader, desc="Validating", leave=True):
            vimages = vimages.to(device).float()
            vmasks = vmasks.to(device).float()

            with torch.autocast(device_type="cuda"):
                voutputs = model(vimages)
                vloss = loss_fn(voutputs, vmasks)

            running_vloss += vloss.item() * vimages.size(0)
            val_batch_losses.append(vloss.item())

    epoch_vloss = running_vloss / len(val_loader.dataset)
    val_losses.append(epoch_vloss)

    print(f"Epoch val loss: {epoch_vloss:.6f}")

    # Track best model
    if epoch_vloss < best_val_loss:
        best_val_loss = epoch_vloss
        print(f"✓ Best validation loss! Saving checkpoint...")

    end_time = time.time()
    epoch_time = end_time - start_time

    # Summary
    print(f"\n[SUMMARY]")
    print(f"Train Loss: {train_loss:.6f} | Val Loss: {epoch_vloss:.6f} | Time: {epoch_time:.2f}s")
    print(f"Loss improvement: {((train_losses[0] - train_loss) / train_losses[0] * 100) if epoch > 0 else 0:.2f}%")

    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated(device) / 1024**3
        reserved = torch.cuda.memory_reserved(device) / 1024**3
        print(f"GPU Memory - Allocated: {allocated:.2f} GB, Reserved: {reserved:.2f} GB")

print(f"\n{'='*80}")
print("Training completed!")
print(f"Final train loss: {train_losses[-1]:.6f}")
print(f"Final val loss: {val_losses[-1]:.6f}")
print(f"Best val loss: {best_val_loss:.6f}")
print(f"{'='*80}")

Using device: cuda
Device: NVIDIA GeForce RTX 3070
Total GPU memory: 8.00 GB

Model: UNet with ResNet18 encoder
Total parameters: 42,611,121
Trainable parameters: 42,611,121
Loss function: Dice Loss (Binary mode)
Optimizer: Adam (lr=1e-4)
Batch size: 4 volumes
Training samples: 36 volumes
Validation samples: 24 volumes

Testing first batch...
Input shape: torch.Size([4, 1, 128, 128, 128])
Input range: [-0.000, 1.000]

Mask shape: torch.Size([4, 1, 128, 128, 128])
Mask range: [0.000, 1.000]
Mask positive pixels: 655446 / 8388608 (7.81%)

Output shape: torch.Size([4, 1, 128, 128, 128])
Output range: [-1.193, 4.445]
Output mean: 0.460

Test loss: 0.853851


Epoch 1/10

[TRAINING]


Using device: cuda
Device: NVIDIA GeForce RTX 3070
Total GPU memory: 8.00 GB

Model: UNet with ResNet18 encoder
Total parameters: 42,611,121
Trainable parameters: 42,611,121
Loss function: Dice Loss (Binary mode)
Optimizer: Adam (lr=1e-4)
Batch size: 4 volumes
Training samples: 36 volumes
Validation samples: 24 volumes

Testing first batch...
Input shape: torch.Size([4, 1, 128, 128, 128])
Input range: [-0.000, 1.000]

Mask shape: torch.Size([4, 1, 128, 128, 128])
Mask range: [0.000, 1.000]
Mask positive pixels: 655446 / 8388608 (7.81%)

Output shape: torch.Size([4, 1, 128, 128, 128])
Output range: [-1.193, 4.445]
Output mean: 0.460

Test loss: 0.853851


Epoch 1/10

[TRAINING]


Training:   0%|          | 0/9 [00:00<?, ?it/s]

Using device: cuda
Device: NVIDIA GeForce RTX 3070
Total GPU memory: 8.00 GB

Model: UNet with ResNet18 encoder
Total parameters: 42,611,121
Trainable parameters: 42,611,121
Loss function: Dice Loss (Binary mode)
Optimizer: Adam (lr=1e-4)
Batch size: 4 volumes
Training samples: 36 volumes
Validation samples: 24 volumes

Testing first batch...
Input shape: torch.Size([4, 1, 128, 128, 128])
Input range: [-0.000, 1.000]

Mask shape: torch.Size([4, 1, 128, 128, 128])
Mask range: [0.000, 1.000]
Mask positive pixels: 655446 / 8388608 (7.81%)

Output shape: torch.Size([4, 1, 128, 128, 128])
Output range: [-1.193, 4.445]
Output mean: 0.460

Test loss: 0.853851


Epoch 1/10

[TRAINING]


Training:   0%|          | 0/9 [00:00<?, ?it/s]

...

Epoch train loss: 0.800778
Learning rate: 1.00e-04

[VALIDATION]


Validating:   0%|          | 0/6 [00:00<?, ?it/s]

Epoch val loss: 0.820314
✓ Best validation loss! Saving checkpoint...

[SUMMARY]
Train Loss: 0.800778 | Val Loss: 0.820314 | Time: 184.38s
Loss improvement: 3.92%
GPU Memory - Allocated: 3.67 GB, Reserved: 13.73 GB

Epoch 4/10

[TRAINING]


Training:   0%|          | 0/9 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# Save the trained model
model_save_path = './unet_plus_plus_3d_resnet34_lctsc.pth'
model.save_pretrained(model_save_path)